[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/praghav444/CE486-586-GIS-Tutorials/blob/main/tutorials/hydrology_geoai/Lab21_22_IUH_GeoAI_Colab.ipynb)

# Labs 21-22: from a DEM to an IUH to GeoAI
**CE 486/586 GIS · Google Colab · Carroll Creek, Tuscaloosa County**

First we redo Tuesday's ArcGIS clicks as code; then we ask whether AI can learn where wetlands are, and whether **hydrology** helps it.

| Part | What we do | Time |
|---|---|---|
| 0 | Set up Colab (GPU) and install the tools | 3 min |
| 1 | **Lab 21 in Python**: DEM → Fill → Flow Direction → Flow Accumulation → Watershed → Streams → Velocity → Travel time → IUH | 12 min |
| 2 | Beautiful maps with **leafmap**: wetlands and aerial photos from 2011 and 2025 | 7 min |
| 3 | Use a **pretrained AI**: find every lake and pond in an aerial photo | 6 min |
| 4 | **Train your own AI** (a U-Net) to find wetlands, then test it somewhere new | 11 min |
| 5 | Give the AI some **hydrology** (HAND, one more tool on Part 1's results) and test again | 9 min |

Tools: [WhiteboxTools](https://www.whiteboxgeo.com) (the same hydrology tools as ArcGIS Pro), [leafmap](https://leafmap.org) and [geoai](https://opengeoai.org) by Qiusheng Wu ([gishub.org](https://gishub.org)). Data: USGS 3DEP elevation and Watershed Boundary Dataset (same as Tuesday), USDA NAIP aerial photos from Microsoft Planetary Computer, USFWS National Wetlands Inventory. **No downloads, no accounts besides Google.**

Run each cell with **Shift + Enter**, from top to bottom.

## Part 0: set up
1. **Runtime ▸ Change runtime type ▸ T4 GPU ▸ Save.** (Training the AI in Parts 4–5 needs the GPU.)
2. Run the install cell below. It takes 2–3 minutes. If Colab asks to **restart the session**, click *Restart*, then go on with the *next* cell (don't rerun the install).

In [ ]:
%pip install -q geoai-py==0.43.1 leafmap==0.64.7 whitebox==2.3.6 omniwatermask==0.6.2 smoothify

In [ ]:
# ---- imports and helper functions: just run this cell -------------------------------------
import os, time, warnings, requests
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt, rasterio
from rasterio.merge import merge
from rasterio.warp import reproject, Resampling
from rasterio.features import rasterize
from shapely.geometry import Point
from pyproj import Transformer
import pystac_client, planetary_computer
import torch, geoai, whitebox
import leafmap.foliumap as leafmap
warnings.filterwarnings("ignore")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none: Runtime > Change runtime type > T4 GPU")

wbt = whitebox.WhiteboxTools()                           # downloads the tools the first time (a few seconds)
wbt.set_working_dir(os.getcwd()); wbt.set_verbose_mode(False)

UTM = 26916                                              # NAD83 / UTM zone 16N, metres
to_ll = Transformer.from_crs(UTM, 4326, always_xy=True).transform
STAC = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1",
                                 modifier=planetary_computer.sign_inplace)

def ll_box(b):
    """UTM box (xmin, ymin, xmax, ymax) -> lon/lat box."""
    (x0, y0), (x1, y1) = to_ll(b[0], b[1]), to_ll(b[2], b[3])
    return [x0, y0, x1, y1]

def get_naip(name, box, year="2025", res=1.0):
    """Read NAIP (R, G, B, near-infrared) for a UTM box straight from the cloud, resampled to res metres."""
    items = list(STAC.search(collections=["naip"], bbox=ll_box(box), datetime=year).items())
    arr, tr = merge([i.assets["image"].href for i in items], bounds=box, res=res)
    path = f"naip_{name}.tif"
    with rasterio.open(path, "w", driver="GTiff", height=arr.shape[1], width=arr.shape[2], count=arr.shape[0],
                       dtype=arr.dtype, crs=f"EPSG:{UTM}", transform=tr, compress="deflate") as d:
        d.write(arr)
    print(f"{path}: {arr.shape[2]} x {arr.shape[1]} pixels at {res} m, from {len(items)} NAIP tile(s)")
    return path

def naip_tiles(box, year):
    """Web-map tile URLs for the NAIP photos covering a box (served by Planetary Computer)."""
    items = STAC.search(collections=["naip"], bbox=ll_box(box), datetime=year).items()
    return [leafmap.stac_tile(collection="naip", item=i.id, assets=["image"], titiler_endpoint="planetary-computer")
            for i in items]

def naip_tile_at(lon, lat, year):
    """Web-map tile URL for the NAIP photo of a given year that contains a point."""
    item = next(STAC.search(collections=["naip"], intersects={"type": "Point", "coordinates": [lon, lat]}, datetime=year).items())
    return leafmap.stac_tile(collection="naip", item=item.id, assets=["image"], titiler_endpoint="planetary-computer")

def get_nwi(box):
    """National Wetlands Inventory polygons for a UTM box (USFWS web service)."""
    url = "https://fwspublicservices.wim.usgs.gov/wetlandsmapservice/rest/services/Wetlands/MapServer/0/query"
    q = dict(geometry=",".join(map(str, ll_box(box))), geometryType="esriGeometryEnvelope", inSR=4326, outSR=4326,
             outFields="ATTRIBUTE,WETLAND_TYPE", f="geojson")
    gdf = gpd.GeoDataFrame.from_features(requests.get(url, params=q, timeout=120).json()["features"], crs=4326)
    return gdf.to_crs(UTM)

def show_rgb(ax, path, title=""):
    with rasterio.open(path) as s:
        img = s.read([1, 2, 3], out_shape=(3, s.height // 4, s.width // 4)); b = s.bounds
    ax.imshow(np.moveaxis(img, 0, -1), extent=(b.left, b.right, b.bottom, b.top))
    ax.set_title(title); ax.set_axis_off()

def add_png(m, path, name, cmap="viridis", vmin=None, vmax=None, opacity=0.7):
    """Put a single-band raster on a leafmap as a coloured image (no tile server needed)."""
    import folium, matplotlib
    with rasterio.open(path) as s:
        dst = np.full((s.height // 2, s.width // 2), np.nan, np.float32)
        x0, y0, x1, y1 = rasterio.warp.transform_bounds(s.crs, 4326, *s.bounds)
        reproject(rasterio.band(s, 1), dst, dst_transform=rasterio.transform.from_bounds(x0, y0, x1, y1, *dst.shape[::-1]),
                  dst_crs=4326, resampling=Resampling.nearest, src_nodata=s.nodata, dst_nodata=np.nan)
    rgba = matplotlib.colormaps[cmap](matplotlib.colors.Normalize(vmin, vmax)(dst))
    rgba[..., 3] = np.where(np.isnan(dst), 0, opacity)
    folium.raster_layers.ImageOverlay(rgba, bounds=[[y0, x0], [y1, x1]], name=name, mercator_project=True).add_to(m)

def score(pred_tif, truth_gdf):
    """Compare a predicted wetland raster (1 = wetland) with NWI polygons."""
    with rasterio.open(pred_tif) as s:
        p = s.read(1) == 1
        t = rasterize(truth_gdf.geometry, out_shape=p.shape, transform=s.transform, fill=0, default_value=1) == 1
    tp, fp, fn = (p & t).sum(), (p & ~t).sum(), (~p & t).sum()
    return {"accuracy": np.mean(p == t), "IoU": tp / max(tp + fp + fn, 1),
            "predicted wetland": p.mean(), "NWI wetland": t.mean()}

def read(path):
    """Read band 1 of a raster as a float array, NoData -> NaN."""
    with rasterio.open(path) as s:
        return s.read(1, masked=True).astype("float32").filled(np.nan)

def save(arr, path, like="dem.tif", nodata=-32768):
    """Write an array as a GeoTIFF on the same grid as `like`, NaN -> NoData."""
    with rasterio.open(like) as s:
        prof = s.profile
    prof.update(dtype="float32", count=1, nodata=nodata, compress="deflate")
    with rasterio.open(path, "w", **prof) as d:
        d.write(np.where(np.isnan(arr), nodata, arr).astype("float32"), 1)
print("Helpers ready.")

## Part 1: Lab 21 in Python (where does the water go, and how long does it take?)
On Tuesday we clicked through ArcGIS Pro: **DEM → Fill → Flow Direction → Flow Accumulation → Snap Pour Point → Watershed → Streams → Velocity → Flow Length (travel time) → Isochrones → IUH**.

Here is the same thing as code. Each ArcGIS tool is one line of **WhiteboxTools** (`wbt.`), a free GIS engine with the same hydrology tools; the Raster Calculator steps are plain NumPy.

| ArcGIS Pro tool | Python |
|---|---|
| Fill | `wbt.fill_depressions` |
| Flow Direction (D8) | `wbt.d8_pointer` |
| Flow Accumulation | `wbt.d8_flow_accumulation` |
| Watershed | `wbt.watershed` |
| Raster Calculator: `Con("fac" > 10000, 1)`, velocity, 1/velocity | NumPy: `fac > 10000`, `np.where`, `1 / vel` |
| Flow Length (DOWNSTREAM, weight) | `wbt.downslope_flowpath_length` |
| Int(travel time / Δt) + Calculate Field | NumPy: `np.bincount` |

**Step 1:** the same data as Tuesday, straight from USGS: the HUC-12 boundary and a 10 m 3DEP DEM (watershed + 1 km).

In [ ]:
# Carroll Creek watershed (HUC-12 031601120412), USGS Watershed Boundary Dataset, in UTM metres
r = requests.get("https://hydro.nationalmap.gov/arcgis/rest/services/wbd/MapServer/6/query",
                 params={"where": "huc12='031601120412'", "outFields": "huc12,name,areasqkm", "outSR": UTM, "f": "geojson"},
                 timeout=60)
ws = gpd.GeoDataFrame.from_features(r.json()["features"], crs=UTM)
WS_BOX = tuple(ws.total_bounds)

# 10 m DEM from the USGS 3DEP image service, snapped to a 10 m grid
CELL = 10
x0, y0 = CELL * ((WS_BOX[0] - 1000) // CELL), CELL * ((WS_BOX[1] - 1000) // CELL)
x1, y1 = CELL * ((WS_BOX[2] + 1000) // CELL + 1), CELL * ((WS_BOX[3] + 1000) // CELL + 1)
ncol, nrow = int((x1 - x0) / CELL), int((y1 - y0) / CELL)
r = requests.get("https://elevation.nationalmap.gov/arcgis/rest/services/3DEPElevation/ImageServer/exportImage",
                 params=dict(bbox=f"{x0},{y0},{x1},{y1}", bboxSR=UTM, imageSR=UTM, size=f"{ncol},{nrow}", format="tiff",
                             pixelType="F32", noDataInterpretation="esriNoDataMatchAny", interpolation="RSP_BilinearInterpolation", f="image"), timeout=300)
with rasterio.MemoryFile(r.content) as mf, mf.open() as s:
    z = s.read(1)
with rasterio.open("dem.tif", "w", driver="GTiff", height=nrow, width=ncol, count=1, dtype="float32", crs=f"EPSG:{UTM}",
                   transform=rasterio.transform.from_origin(x0, y1, CELL, CELL), nodata=-32768) as d:
    d.write(z.astype("float32"), 1)

dem = read("dem.tif")
print(ws[["name", "areasqkm"]].to_string(index=False))
print(f"DEM: {ncol} x {nrow} cells of {CELL} m, elevation {np.nanmin(dem):.1f} to {np.nanmax(dem):.1f} m")

**Part A: where does the water go?** Fill → Flow Direction → Flow Accumulation → outlet → Watershed.

In [ ]:
# A1  Fill: remove pits so water cannot get stuck
wbt.fill_depressions("dem.tif", "fill.tif", fix_flats=True)
fill_depth = read("fill.tif") - dem
print(f"A1 Fill done. Max fill depth = {np.nanmax(fill_depth):.1f} m")

# A2  Flow Direction (D8, with ArcGIS codes 1, 2, 4, ... 128)
wbt.d8_pointer("fill.tif", "fdr.tif", esri_pntr=True)
codes = np.unique(read("fdr.tif")); print("A2 Flow Direction done. Codes:", codes[codes > 0].astype(int))

# A3  Flow Accumulation: how many cells drain through each cell
wbt.d8_flow_accumulation("fdr.tif", "fac.tif", out_type="cells", pntr=True, esri_pntr=True)
fac = read("fac.tif")
print(f"A3 Flow Accumulation done. Max = {np.nanmax(fac):,.0f} cells")

# A4  Outlet: the cell inside the official boundary with the largest flow accumulation
with rasterio.open("dem.tif") as s:
    T = s.transform
    inside = rasterize(ws.geometry, out_shape=s.shape, transform=T) == 1
row, col = np.unravel_index(np.nanargmax(np.where(inside, fac, -1)), fac.shape)
ox, oy = T * (col + 0.5, row + 0.5)
gpd.GeoDataFrame(geometry=[Point(ox, oy)], crs=UTM).to_file("outlet.shp")

# A5  Watershed: every cell that drains to the outlet
wbt.watershed("fdr.tif", "outlet.shp", "watershed.tif", esri_pntr=True)
ws_mask = read("watershed.tif") == 1
A_km2 = ws_mask.sum() * CELL * CELL / 1e6
print(f"A5 Watershed: {ws_mask.sum():,} cells = {A_km2:.2f} km2   (official WBD: {ws.area.sum() / 1e6:.2f} km2)")

**Part B: how long does it take?** Streams → Velocity → Weight = 1/velocity → weighted Flow Length = travel time → isochrones → IUH.

In [ ]:
# Model choices: change these and rerun to ask "what if?"
V_HILL = 0.5          # hillslope velocity, m/s
V_STREAM = 1.0        # channel velocity, m/s
THRESH_CELLS = 10000  # stream threshold: 10,000 cells x 100 m2 = 1 km2
DT_H = 0.25           # time step for the isochrone bands, hours (15 min)

# B1  Streams: cells that drain more than 1 km2  (Raster Calculator: Con("fac" > 10000, 1))
streams = fac > THRESH_CELLS
save(np.where(streams, 1.0, 0.0), "streams.tif")

# B2-B3  Velocity and weight = 1 / velocity (seconds per metre)
velocity = np.where(streams, V_STREAM, V_HILL)
save(1.0 / velocity, "weight.tif")

# B4-B6  Weighted downstream Flow Length to OUR outlet (the watershed grid stops it there) = travel time
save(np.where(ws_mask, 1.0, np.nan), "ws01.tif")
wbt.downslope_flowpath_length("fdr.tif", "ttime_s.tif", watersheds="ws01.tif", weights="weight.tif", esri_pntr=True)
ttime_h = np.where(ws_mask, read("ttime_s.tif") / 3600.0, np.nan)
save(ttime_h, "ttime_h.tif")
print(f"B1-B6 done. Time of concentration (max travel time) = {np.nanmax(ttime_h):.2f} h")

# B7-B8  Isochrone bands of DT_H hours, area per band, IUH = dA / (A dt)
zone = (ttime_h[ws_mask] // DT_H).astype(int)               # 0, 1, 2, ... (ArcGIS: Int(ttime_h / 0.25) + 1)
cells = np.bincount(zone)
time_h = (np.arange(len(cells)) + 0.5) * DT_H                # band midpoints, h
area_km2 = cells * CELL * CELL / 1e6
iuh = area_km2 / (area_km2.sum() * DT_H)                     # 1/h
print(f"B7 {len(cells)} bands.  B8 sum of areas = {area_km2.sum():.2f} km2, sum(IUH) x dt = {iuh.sum() * DT_H:.3f}")
print(f"IUH peak = {iuh.max():.3f} 1/h at t = {time_h[iuh.argmax()]:.3f} h;  mean travel time = {np.sum(time_h * iuh * DT_H):.2f} h")
pd.DataFrame({"band": np.arange(1, len(cells) + 1), "t mid (h)": time_h, "area (km2)": area_km2.round(3),
              "IUH (1/h)": iuh.round(4)}).head(8)

In [ ]:
# What each step produced (red line = watershed, star = outlet)
from scipy.ndimage import binary_dilation
from matplotlib.colors import ListedColormap
d8 = np.log2(read("fdr.tif"))                              # D8 codes 1, 2, 4 ... 128 -> 0, 1, 2 ... 7 (E, SE, S ... NE)
wide_streams = binary_dilation(streams & ws_mask, iterations=2)  # thicker, so the 1-cell streams are visible
panels = [(dem, "DEM (m)", "terrain", None, None),
          (np.where(fill_depth > 0.05, fill_depth, np.nan), "A1  Fill depth (m)", "magma_r", 0, 3),
          (d8, "A2  Flow direction (0 = E, 2 = S, 4 = W, 6 = N)", ListedColormap(plt.cm.Set1.colors[:8]), -0.5, 7.5),
          (np.log10(fac + 1), "A3  Flow accumulation (log10 cells)", "Blues", 0.5, 5),
          (np.where(ws_mask, np.where(wide_streams, V_STREAM, V_HILL), np.nan), "B1-B2  Streams and velocity (m/s)", "coolwarm", None, None),
          (ttime_h, "B6  Travel time to outlet (h)", "viridis_r", None, None)]
fig, ax = plt.subplots(2, 3, figsize=(17, 9))
for a, (zz, title, cmap, vmin, vmax) in zip(ax.flat, panels):
    im = a.imshow(zz, cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
    a.contour(ws_mask, levels=[0.5], colors="red", linewidths=0.8)
    a.plot(col, row, "r*", ms=12); a.set_title(title); a.set_axis_off(); fig.colorbar(im, ax=a, shrink=0.7)
plt.tight_layout(); plt.show()

# IUH: bars from GIS + a smooth line through them
from scipy.interpolate import PchipInterpolator      # smooth, never negative, no overshoot
t_end = time_h.max() + DT_H / 2
tt = np.linspace(0, t_end, 300)
smooth = PchipInterpolator(np.r_[0, time_h, t_end], np.r_[0, iuh, 0])(tt)
plt.figure(figsize=(10, 4.5))
plt.bar(time_h, iuh, width=DT_H, color="lightgrey", edgecolor="black", label=f"IUH ({DT_H * 60:.0f}-min bands)")
plt.plot(tt, smooth, color="#C00000", lw=2.5, label="smooth line")
plt.xlabel("Travel time to outlet, t (h)"); plt.ylabel("h(t)  (1/h)")
plt.title(f"Carroll Creek IUH   A = {A_km2:.2f} km2,  peak {iuh.max():.2f} 1/h at {time_h[iuh.argmax()]:.2f} h")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# The travel-time map on a real map: zoom in and switch layers (top right)
m = leafmap.Map(center=[33.33, -87.61], zoom=12)
m.add_basemap("SATELLITE")
add_png(m, "ttime_h.tif", "Travel time to outlet (h)", cmap="viridis_r", vmin=0, vmax=8, opacity=0.75)
save(np.where(streams & ws_mask, 1.0, np.nan), "streams_ws.tif")
add_png(m, "streams_ws.tif", "Streams (> 1 km2)", cmap="cool", vmin=0, vmax=1, opacity=1)
m.add_gdf(ws.to_crs(4326), layer_name="Official WBD boundary", zoom_to_layer=False,
          style={"color": "yellow", "weight": 3, "fillOpacity": 0})
m.add_marker(location=[to_ll(ox, oy)[1], to_ll(ox, oy)[0]], tooltip="Outlet")
m.add_colorbar(colors=["#fde725", "#21918c", "#440154"], vmin=0, vmax=8, caption="Travel time to outlet (h)")
m

**Compare with Tuesday** (ArcGIS Pro): watershed 66.47 km², time of concentration 8.0 h, IUH peak 0.26 1/h at 6.4 h. Python gives almost the same numbers; small differences come from how each program routes water across the flat lake surface.

**Optional (homework): from the IUH to a unit hydrograph and a flood.** 1 cm of excess rain on A km² is A × 0.01 m³ of water; the IUH says what share of it arrives each hour. Spread the rain over D hours (a moving average) and you have the textbook D-hour unit hydrograph.

In [ ]:
A_m2 = A_km2 * 1e6
U0 = A_m2 * 0.01 / 3600 * iuh                              # instantaneous UH, m3/s per cm
plt.figure(figsize=(10, 4.5))
for D, style in [(1, "-"), (3, "--"), (6, ":")]:
    n = int(round(D / DT_H))
    U = np.r_[0, np.convolve(U0, np.ones(n) / n)]          # D-hour UH = moving average over D hours
    t = np.arange(len(U)) * DT_H
    plt.plot(t, U, "k" + style, lw=2, label=f"{D}-h UH: peak {U.max():.1f} m3/s per cm at {t[U.argmax()]:.1f} h")
plt.xlabel("Time (h)"); plt.ylabel("Q (m3/s per cm of excess rain)")
plt.title("Same 1 cm of rain, spread over a longer time: lower, later, wider"); plt.legend(); plt.show()

# A storm: 1.0, 2.5 and 0.5 cm of excess rain in three hours -> scale, shift, add (convolution)
P = np.repeat([1.0, 2.5, 0.5], int(1 / DT_H)) * DT_H        # cm per time step
Q = np.convolve(P, U0)
print(f"Storm hydrograph peak = {Q.max():.0f} m3/s at {(Q.argmax() + 1) * DT_H:.2f} h")

## Part 2: beautiful maps with leafmap
Now the GeoAI question: **can AI learn where wetlands are?** First, look at the data, live from public web services: the **National Wetlands Inventory** (US Fish & Wildlife Service) and **NAIP** aerial photos (USDA, 30 cm, summer 2025).

In [ ]:
nwi = gpd.clip(get_nwi(WS_BOX), ws)
print(nwi.WETLAND_TYPE.value_counts().to_string())

In [ ]:
COLORS = {"Freshwater Forested/Shrub Wetland": "#39e75f", "Freshwater Emergent Wetland": "#ffff66",
          "Freshwater Pond": "#00bfff", "Lake": "#1e5aff", "Riverine": "#00ffff"}
m = leafmap.Map(center=[33.33, -87.61], zoom=12)
m.add_basemap("SATELLITE")
for url in naip_tiles(WS_BOX, "2025"):
    m.add_tile_layer(url, name="NAIP 2025 (30 cm)", attribution="USDA NAIP / Microsoft Planetary Computer")
for t, col in COLORS.items():
    sub = nwi[nwi.WETLAND_TYPE == t]
    if len(sub):
        m.add_gdf(sub.to_crs(4326), layer_name=t, zoom_to_layer=False,
                  style={"color": col, "fillColor": col, "weight": 1, "fillOpacity": 0.6})
m.add_gdf(ws.to_crs(4326), layer_name="Carroll Creek watershed", zoom_to_layer=False,
          style={"color": "yellow", "weight": 3, "fillOpacity": 0})
m.add_legend(title="NWI wetlands", legend_dict=COLORS)
m      # zoom in (the 30 cm NAIP photo appears from zoom 14); the layer button (top right) switches layers

**Look:** where do the forested wetlands (dark green) sit? Zoom in on one. Can you see it in the photo, or does it just look like forest?

**Split map:** the same view in 2011 and 2025. Drag the slider. What changed around Lake Tuscaloosa?

In [ ]:
LAKE_BOX = (447000, 3684500, 450000, 3687500)          # the Carroll Creek arm of Lake Tuscaloosa (UTM, m)
lon, lat = -87.540, 33.300                            # map centre: Carroll Creek arm of the lake
m = leafmap.Map(center=[lat, lon], zoom=16)
m.split_map(left_layer=naip_tile_at(lon, lat, "2011"), right_layer=naip_tile_at(lon, lat, "2025"),
            left_label="NAIP 2011 (1 m)", right_label="NAIP 2025 (30 cm)")
m

## Part 3: a pretrained AI finds the water
Someone else already trained a deep-learning model (**OmniWaterMask**) on thousands of images. We just *use* it: no labels, no training. It looks at the red, green, blue and **near-infrared** bands (water absorbs near-infrared, so it is dark).

![How OmniWaterMask works](https://raw.githubusercontent.com/praghav444/CE486-586-GIS-Tutorials/main/figures/OmniWaterMask_AI_Segmentation_Architecture.png)

**How it works** (optional reading):
1. **Input:** the photo's 4 bands, the NDWI water index computed from them, and mapped water, roads and buildings from OpenStreetMap.
2. **U-Net:** the photo is cut into patches. The left half of the "U" shrinks each patch into deep features (*what* is there); the right half grows them back to full size (*where* it is), giving a water probability for every pixel. In Part 4 you will train a U-Net yourself.
3. **Fusion:** the U-Net probabilities are combined with NDWI and the mapped polygons (roads and buildings help remove false water), giving the final water mask.

We read a 3 km × 3 km piece of NAIP at 2 m (the next cell takes under a minute on the GPU).

In [ ]:
lake_tif = get_naip("lake", LAKE_BOX, res=2.0)
torch.cuda.empty_cache()
t0 = time.time()
water = geoai.segment_water(lake_tif, band_order="naip", output_raster="water_mask.tif",
                            output_vector="water.geojson", smooth=True, min_size=100, verbose=False,
                            batch_size=1)      # one patch at a time: ~5 GB of GPU memory (the default needs ~20 GB)
water = water.to_crs(UTM)
print(f"Found {len(water)} water bodies, {water.area.sum() / 1e6:.2f} km2, in {time.time() - t0:.0f} s")
water.sort_values("area_m2", ascending=False)[["area_m2", "perimeter_m", "elongation"]].head()

In [ ]:
m = leafmap.Map(center=[33.296, -87.556], zoom=14)
m.add_basemap("SATELLITE")
for url in naip_tiles(LAKE_BOX, "2025"):
    m.add_tile_layer(url, name="NAIP 2025", attribution="USDA NAIP")
m.add_gdf(water.to_crs(4326), layer_name="AI: water", zoom_to_layer=False,
          style={"color": "cyan", "fillColor": "cyan", "weight": 2, "fillOpacity": 0.3})
lake_nwi = get_nwi(LAKE_BOX)
m.add_gdf(lake_nwi[lake_nwi.WETLAND_TYPE.isin(["Lake", "Freshwater Pond"])].to_crs(4326), layer_name="NWI: lake + ponds",
          zoom_to_layer=False, style={"color": "red", "weight": 1.5, "fillOpacity": 0})
m.add_legend(legend_dict={"AI water": "#00ffff", "NWI lake / ponds": "#ff0000"})
m

**Think:** did the AI find ponds that NWI missed (or that were dug after NWI was mapped)? Water is easy: it looks the same everywhere. **Wetlands are harder**: most of Carroll Creek's wetlands are *forested*. From above they look like... forest.

## Part 4: train your own AI to find wetlands
No pretrained model knows Alabama's bottomland wetlands, so we **train one**: a **U-Net**, the classic deep-learning model for maps. It looks at the photo, pixel by pixel, and learns which patterns mean *wetland*.

* **Examples (labels):** NWI vegetated wetlands.
* **Inputs:** the 4 NAIP bands at 1 m.
* **Train** upstream, **test** downstream, on ground the model has never seen.

In [ ]:
TRAIN_BOX = (437500, 3687500, 440500, 3690500)     # upstream, 3 km x 3 km
TEST_BOX = (442500, 3683500, 445500, 3686500)      # downstream
VEG = ["Freshwater Forested/Shrub Wetland", "Freshwater Emergent Wetland"]

labels = {}
for name, box in [("train", TRAIN_BOX), ("test", TEST_BOX)]:
    get_naip(name, box, res=1.0)
    w = get_nwi(box); w = gpd.clip(w[w.WETLAND_TYPE.isin(VEG)], list(box)); w["class"] = 1
    w.to_file(f"wet_{name}.geojson"); labels[name] = w

fig, ax = plt.subplots(1, 3, figsize=(17, 6))
ws.boundary.plot(ax=ax[0], color="k"); nwi[nwi.WETLAND_TYPE.isin(VEG)].plot(ax=ax[0], color="green")
for b, c, lab in [(TRAIN_BOX, "blue", "train"), (TEST_BOX, "red", "test")]:
    ax[0].add_patch(plt.Rectangle(b[:2], b[2] - b[0], b[3] - b[1], fill=False, ec=c, lw=2.5)); ax[0].text(b[0], b[3] + 200, lab, color=c)
ax[0].set_title("Where we train and test"); ax[0].set_axis_off()
for a, name in zip(ax[1:], ["train", "test"]):
    show_rgb(a, f"naip_{name}.tif", f"NAIP, {name} area (outline = NWI wetland)")
    labels[name].boundary.plot(ax=a, color="yellow", lw=1)
plt.tight_layout(); plt.show()

In [ ]:
# Cut the training photo into 512 x 512 "chips" with matching label masks
geoai.export_geotiff_tiles(in_raster="naip_train.tif", out_folder="chips_img", in_class_data="wet_train.geojson",
                           tile_size=512, stride=256, quiet=True)
print(len(os.listdir("chips_img/images")), "training chips")

EPOCHS = 15            # one epoch = the model sees every chip once
t0 = time.time()
geoai.train_segmentation_model(images_dir="chips_img/images", labels_dir="chips_img/labels", output_dir="unet_img",
                               architecture="unet", encoder_name="resnet34", encoder_weights="imagenet",
                               num_channels=4, num_classes=2, batch_size=8, num_epochs=EPOCHS,
                               learning_rate=0.001, val_split=0.2, verbose=False)
print(f"Trained in {time.time() - t0:.0f} s")

In [ ]:
results = {}
for name in ["train", "test"]:
    geoai.semantic_segmentation(input_path=f"naip_{name}.tif", output_path=f"pred_img_{name}.tif",
                                model_path="unet_img/best_model.pth", architecture="unet", encoder_name="resnet34",
                                num_channels=4, num_classes=2, window_size=512, overlap=256, batch_size=8, quiet=True)
    results[("Photo only", name)] = score(f"pred_img_{name}.tif", labels[name])
pd.DataFrame(results).T.round(2)

**IoU** (intersection over union) = overlap between the AI's wetlands and NWI's, divided by their union: 1 is perfect, 0 is no overlap. It is much stricter than accuracy.

**What happened on the test area?** Look at both rows. Then look at *accuracy*: if wetlands cover only 14% of the test area, a model that says "no wetland anywhere" still scores 86%.

**Why?** Under a tree canopy, a wetland looks like any other forest. The photo alone doesn't tell the model *where water collects*.

## Part 5: give the AI some hydrology
**HAND** (height above nearest drainage) = how high each cell sits above the stream it drains to. Wetlands sit low, close to the water table. We already have everything from Part 1 (the filled DEM and the 1 km² streams), so HAND is **one more tool**.

In [ ]:
# HAND: follow each cell's flow path down to the first stream cell and subtract the elevations
wbt.elevation_above_stream("fill.tif", "streams.tif", "hand.tif")
hand = read("hand.tif")
save(np.where(ws_mask, hand, np.nan), "hand_ws.tif")          # a copy clipped to the watershed, for the map
print(f"HAND median {np.nanmedian(hand[ws_mask]):.1f} m; {np.mean(hand[ws_mask] < 2):.0%} of the watershed is less than 2 m above a stream")

In [ ]:
m = leafmap.Map(center=[33.33, -87.61], zoom=12)
m.add_basemap("SATELLITE")
add_png(m, "hand_ws.tif", "HAND (m)", cmap="viridis_r", vmin=0, vmax=20)
m.add_gdf(nwi[nwi.WETLAND_TYPE.isin(VEG)].to_crs(4326), layer_name="NWI wetlands", zoom_to_layer=False,
          style={"color": "red", "weight": 1, "fillOpacity": 0})
m.add_gdf(ws.to_crs(4326), layer_name="Watershed", zoom_to_layer=False, style={"color": "white", "weight": 2, "fillOpacity": 0})
m.add_colorbar(colors=["#fde725", "#21918c", "#440154"], vmin=0, vmax=20, caption="HAND (m): yellow = low")
m

In [ ]:
# Add HAND as a 5th band to each photo (0-25.5 m stored as 0-255)
for name in ["train", "test"]:
    with rasterio.open(f"naip_{name}.tif") as s, rasterio.open("hand.tif") as h:
        hb = np.full((s.height, s.width), np.nan, np.float32)
        reproject(rasterio.band(h, 1), hb, dst_transform=s.transform, dst_crs=s.crs, resampling=Resampling.bilinear,
                  dst_nodata=np.nan)
        hb = np.clip(np.nan_to_num(hb, nan=25.5) * 10, 0, 255).astype(np.uint8)
        prof = s.profile; prof.update(count=5)
        with rasterio.open(f"naiphand_{name}.tif", "w", **prof) as o:
            o.write(np.concatenate([s.read(), hb[None]]))

geoai.export_geotiff_tiles(in_raster="naiphand_train.tif", out_folder="chips_hand", in_class_data="wet_train.geojson",
                           tile_size=512, stride=256, quiet=True)
t0 = time.time()
geoai.train_segmentation_model(images_dir="chips_hand/images", labels_dir="chips_hand/labels", output_dir="unet_hand",
                               architecture="unet", encoder_name="resnet34", encoder_weights="imagenet",
                               num_channels=5, num_classes=2, batch_size=8, num_epochs=EPOCHS,
                               learning_rate=0.001, val_split=0.2, verbose=False)
print(f"Trained in {time.time() - t0:.0f} s")
for name in ["train", "test"]:
    geoai.semantic_segmentation(input_path=f"naiphand_{name}.tif", output_path=f"pred_hand_{name}.tif",
                                model_path="unet_hand/best_model.pth", architecture="unet", encoder_name="resnet34",
                                num_channels=5, num_classes=2, window_size=512, overlap=256, batch_size=8, quiet=True)
    results[("Photo + HAND", name)] = score(f"pred_hand_{name}.tif", labels[name])
table = pd.DataFrame(results).T.round(2)
table

In [ ]:
iou = table["IoU"].unstack().loc[["Photo only", "Photo + HAND"], ["train", "test"]]
ax = iou.plot.bar(figsize=(7, 4), rot=0, color=["#8CADAE", "#C00000"])
ax.set_ylabel("IoU with NWI (higher = better)"); ax.set_title("Does the AI transfer to new ground?"); plt.show()

### Where is the AI right, and where is it wrong?
IoU is easier to understand on a map. Every pixel falls in one of three groups:
* **green = hit**: the AI and NWI both say wetland
* **orange = false alarm**: the AI says wetland, NWI does not
* **magenta = missed**: NWI says wetland, the AI does not

**IoU = green ÷ (green + orange + magenta).** Look at the downstream maps: where does the photo-only AI raise false alarms (hint: lawns, parking lots, neighbourhoods)? What happens there once the AI also knows HAND, the height above the nearest stream?

In [ ]:
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap
OUT_CMAP = ListedColormap([(0, 0, 0, 0), "#1a9850", "#ff8c00", "#e7298a"])   # none, hit, false alarm, missed

def outcome(pred_tif, truth_gdf, step=4):
    """Per pixel: 0 = neither, 1 = hit, 2 = false alarm, 3 = missed (on a grid `step` times coarser, for plotting)."""
    with rasterio.open(pred_tif) as s:
        p = s.read(1, out_shape=(s.height // step, s.width // step), resampling=Resampling.nearest) == 1
        tr = s.transform * s.transform.scale(s.width / p.shape[1], s.height / p.shape[0])
    t = rasterize(truth_gdf.geometry, out_shape=p.shape, transform=tr, fill=0, default_value=1) == 1
    return np.select([p & t, p & ~t, ~p & t], [1, 2, 3], 0)

AREA = {"train": "upstream (trained here)", "test": "downstream (never seen)"}
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for i, (model, pre) in enumerate([("Photo only", "pred_img"), ("Photo + HAND", "pred_hand")]):
    for j, name in enumerate(["train", "test"]):
        ax = axes[i, j]
        with rasterio.open(f"naip_{name}.tif") as s:
            img = s.read([1, 2, 3], out_shape=(3, s.height // 4, s.width // 4)); b = s.bounds
        ext = (b.left, b.right, b.bottom, b.top)
        ax.imshow(np.moveaxis(img, 0, -1), extent=ext, alpha=0.55)          # faded photo underneath
        ax.imshow(outcome(f"{pre}_{name}.tif", labels[name]), cmap=OUT_CMAP, vmin=0, vmax=3,
                  extent=ext, interpolation="nearest")
        ax.set_title(f"{model}, {AREA[name]}\nIoU = {table.loc[(model, name), 'IoU']:.2f}")
        ax.set_axis_off()
fig.legend(handles=[Patch(color="#1a9850", label="hit (AI and NWI)"), Patch(color="#ff8c00", label="false alarm (AI only)"),
                    Patch(color="#e7298a", label="missed (NWI only)")], loc="lower center", ncol=3, fontsize=13, frameon=False)
plt.tight_layout(rect=(0, 0.04, 1, 1)); plt.show()

In [ ]:
# Final interactive map of the test area: NWI vs both AIs (switch layers on and off)
m = leafmap.Map(center=[33.31, -87.60], zoom=14)
m.add_basemap("SATELLITE")
for url in naip_tiles(TEST_BOX, "2025"):
    m.add_tile_layer(url, name="NAIP 2025", attribution="USDA NAIP")
for tif, lab, col in [("pred_img_test.tif", "AI: photo only", "#ffa500"), ("pred_hand_test.tif", "AI: photo + HAND", "#00ffff")]:
    g = geoai.raster_to_vector(tif, f"{tif[:-4]}.geojson", min_area=200)
    if len(g) == 0:                       # the AI found no wetland patch bigger than 200 m2 here
        print(f"{lab}: no wetland predicted in the test area")
        continue
    m.add_gdf(g.to_crs(4326), layer_name=lab, zoom_to_layer=False,
              style={"color": col, "fillColor": col, "weight": 1, "fillOpacity": 0.35})
m.add_gdf(labels["test"].to_crs(4326), layer_name="NWI wetlands", zoom_to_layer=False,
          style={"color": "red", "weight": 2, "fillOpacity": 0})
m.add_legend(legend_dict={"AI: photo only": "#ffa500", "AI: photo + HAND": "#00ffff", "NWI wetland": "#ff0000"})
m

## Take-aways
1. **leafmap** turns a few lines of Python into an interactive map of live, public data.
2. **Pretrained AI** works well when the thing looks the same everywhere (water).
3. **Train your own AI** when nobody has a model for your problem, and **test it somewhere new**: on the training area it looked fine.
4. **Hydrology makes the AI transfer.** The photo shows the canopy; HAND shows where the water collects. The best GeoAI models still respect hydrology.

## Try it yourself
* Change `EPOCHS` to 5 or 30. What happens to the photo-only model?
* Swap `TRAIN_BOX` and `TEST_BOX`: train downstream, test upstream.
* In Part 3, move `LAKE_BOX` somewhere else in Alabama (any UTM 16N box) and find the water.
* Replace HAND with the **slope** (`np.gradient` of the DEM). Is HAND better?